# 演示词向量的应用，把训练好的词向量指定为词嵌入层的初始化向量

In [1]:
from torch import nn
import torch
import pandas as pd
import jieba
from gensim.models import KeyedVectors,Word2Vec

In [3]:
#加载词向量
wv=KeyedVectors.load_word2vec_format('./model/自训练词向量.txt')
wv['地铁']

array([ 0.31167284,  0.70360816, -0.29015705,  0.14268799,  0.0013504 ,
       -0.15363438,  0.49773946,  0.25209934, -0.2919922 ,  0.26833546,
       -0.4245242 , -0.09085206,  0.6138091 , -0.1619665 ,  0.13487744,
       -0.02129916,  0.17402878, -0.46644595,  0.20840727, -0.32417634,
        0.6050266 ,  0.02632684,  0.09812877, -0.10442194,  0.05913408,
       -0.13324331,  0.31101927,  0.30814055, -0.09409472,  0.24442194,
        0.5023426 ,  0.27744135, -0.2429727 , -0.8346241 ,  0.5350571 ,
        0.13654959,  0.04348114,  0.44610396, -0.07416711, -0.17176312,
       -0.41295156,  0.1975789 , -0.25746366,  0.25014243,  0.69349194,
       -0.3318286 , -0.03058613,  0.32564712, -0.2386245 ,  0.3918105 ,
       -0.573219  , -0.4559707 , -0.08124863, -0.01379509,  0.46429867,
       -0.19663331,  0.28550416,  0.5328392 , -0.16392483,  0.02070505,
        0.07862406,  0.53772444,  0.0769607 ,  0.19457282, -0.02623541,
        0.51148283,  0.10423505,  0.45533386,  0.12477988,  0.58

In [5]:
#准备词向量矩阵
embedding_size=wv.vector_size
embedding_num=len(wv)
embedding_matrix=torch.randn(embedding_num,embedding_size)
embedding_matrix.shape      #17018个词，每个词向量维度为100

torch.Size([17018, 100])

In [11]:
#填充词向量矩阵
# wv.index_to_key         #index_to_key会获取按照词向量索引排列的词语列表
# wv.key_to_index           #key_to_index可以获取词语及其对应索引构成的词典，key：词语，values：索引
for index,word in enumerate(wv.index_to_key):        #获取每个词
    embedding_matrix[index]=torch.tensor(wv[word])       #取出来的word是列表，要转成张量
embedding_matrix

tensor([[ 0.0768, -0.0143, -0.0634,  ...,  0.0951, -0.1499, -0.3109],
        [ 0.3571, -0.0204,  0.0460,  ...,  0.0796, -0.0983, -0.2832],
        [ 0.0608, -0.0006, -0.1669,  ...,  0.0920, -0.0563,  0.0145],
        ...,
        [-0.0243,  0.1583, -0.0248,  ..., -0.1238, -0.0290, -0.1745],
        [-0.0297,  0.1135, -0.0510,  ..., -0.2181, -0.0538, -0.1763],
        [-0.0072,  0.2032, -0.0018,  ..., -0.2206, -0.0374, -0.2650]])

In [17]:
#创建词嵌入层
embedding=nn.Embedding.from_pretrained(embedding_matrix,freeze=True)        #指定词向量矩阵作为初始参数，冻结冻结词嵌入层的词向量值，使其在训练过程                                                                            中不计算梯度、不更新参数

In [18]:
#测试
# text='我喜欢乘坐宇宙飞船'        #宇宙飞船属于OOV
text='我喜欢乘坐地铁'
tokens=jieba.lcut(text)
token_index=[wv.key_to_index[token] for token in tokens if token in wv.index_to_key]
input_tensor=torch.tensor(token_index)
embedding(input_tensor)         #取出这些词对应的词向量

tensor([[ 0.0742, -0.0446, -0.2337,  0.3156,  0.3238, -0.5041,  0.0981, -0.0642,
         -0.0378,  0.1065,  0.1707, -0.1890,  0.0216, -0.1061, -0.0429, -0.3690,
         -0.0138,  0.1312,  0.1644, -0.5362,  0.2738,  0.1839,  0.1843, -0.2399,
         -0.1934,  0.3157,  0.0530, -0.1619,  0.1855, -0.1445,  0.4279,  0.1496,
         -0.0712, -0.3067, -0.0960, -0.1150,  0.2582, -0.1240, -0.0545, -0.5328,
          0.0448, -0.3095,  0.2523, -0.1696,  0.2534, -0.0500, -0.2384, -0.1751,
          0.2445,  0.1398, -0.0082,  0.0088,  0.2474,  0.1838, -0.1684,  0.1037,
          0.1242, -0.1107, -0.6732,  0.0263,  0.2194, -0.1246, -0.3726,  0.1098,
          0.0510,  0.1297,  0.0223,  0.4653,  0.0483, -0.2632,  0.3126,  0.0109,
          0.3019, -0.1067,  0.2624,  0.3466,  0.3554, -0.1151,  0.2288,  0.0761,
         -0.0069,  0.4647, -0.2877,  0.2236, -0.2367, -0.0962, -0.0529,  0.2782,
          0.1020, -0.1986,  0.3809, -0.5107, -0.4858,  0.2865,  0.4781, -0.0112,
          0.1571, -0.0898, -

In [ ]:
#如果存在oov，即词表中不存在的词，就会发生错误，因此需要维护一份自己的词表
#以宇宙飞船为例
unk_token='宇宙飞船'
# l1=[1,2,3,3,6]
# l2=[unk_token]+l1
#拼接未知词语原词表
new_tokens=[unk_token]+wv.index_to_key
#创建新的词语索引的对应词典
new_dic={token:index for index,token in enumerate(new_tokens)}      #由于宇宙飞船放在第一个，词和对应索引会发生错位


In [ ]:
#准备词向量矩阵
embedding_size=wv.vector_size
embedding_num=len(new_tokens)
embedding_matrix=torch.randn(embedding_num,embedding_size)
embedding_matrix.shape

In [ ]:
#填充矩阵
for index,word in enumerate(new_tokens):
    if word in wv.index_to_key:
        embedding_matrix[index]=torch.tensor(wv[word])